## Three Parts of this lab

### Part1: A simple "Agent" and "Agent Loop"
- Basically an LLM Call. We'll add tracing and streaming to the mix

### Part2: Adding Tool

A familiar one

### Part3: Adding Memory

So that different agent calls know about each other

In [32]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent 
from openai import OpenAI, AsyncOpenAI


from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_default_openai_client, set_default_openai_api, set_trace_processors
from langsmith.integrations.openai_agents_sdk import OpenAIAgentsTracingProcessor

load_dotenv(override=True)

True

In [33]:
#set_tracing_disabled(True)

### Sidenote

The actual name of this framework on the official Python index pipi.org os openai-agents

So for your own projects in the future, you would do:
- pip install openai-agents or  uv add openai-agents
- followed by: from agents import Agent, Runner, trace

- Beware that doing a pip install agents would install something completely different - an older reinforcement learning library.

In [34]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
ollama = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Tell SDK to use ollama client
set_default_openai_client(ollama)

# Tell the SDK to use chat completions
set_default_openai_api("chat_completions")



In [35]:
# Make an agent with name, instructions and model

agent = Agent(
    name="agent",
    instructions="You are a helpful assistant",
    model="qwen2.5:7b",
)

In [36]:
set_trace_processors([OpenAIAgentsTracingProcessor()])

In [37]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [38]:
result.final_output

"Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to eat the lettuce?\n\nBecause it didn't want to download the latest kale updates!"

In [39]:
# The details of the llm calls
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to eat the lettuce?\n\nBecause it didn't want to download the latest kale updates!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-16'}}]

### Adding Observability with a trace


In [40]:
## Trace works only with openai API Key. With Ollama it silently does nothing. Let us use langsmith
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

print(result.final_output)


Sure! Here's a light-hearted joke about Autonomous AI Agents:

Why did the Autonomous AI Agent break up with its partner?

Because it couldn't get a signal on its emotional frequency!


## Adding Tools




In [41]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = os.getenv("PUSHOVER_URL")
if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

if pushover_url:
    print("Pushover URL found and looks good")
else:
    print("Pushover URL not found")




Pushover user found and looks good
Pushover token found and looks good
Pushover URL found and looks good


In [42]:

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [43]:


push("HEY!!")



Push: HEY!!


In [44]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [45]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001EAA9CD0AD0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [46]:
push_tool.description

'Send the given message to the user as a push notification'

In [47]:
notifier = Agent(
    name="notifier",
    instructions="You are a helpful assistant that sends push notifications to the user",
    tools=[push_tool],
    model="qwen2.5:7b",
)

In [48]:
result = await Runner.run(notifier, "Notify the user that the pizza is here")


In [49]:
result.final_output

"You've been notified that your pizza is here! Enjoy your meal!"

## Part3: Sessions (Memory)

- Within a Runner.run() application level turn, the conversation history is maintained.
- But each call to Runner.run() is a fresh start.

In [50]:
agent = Agent(
    name="notifier",
    model="qwen2.5:7b",
)

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Bilal.")
print(response.final_output)

Hello Bilal! Nice to meet you. How can I assist you today? Is there anything specific you're interested in discussing or any questions you have?


In [ ]:
response = await Runner.run(agent, "what is my name")
print(response.final_output)

You didn't provide your name in your question. In our conversation, you are referred to as "User." If you'd like to know more about yourself or discuss something specific, feel free to share more details!


### Approach 1: Manually pass in the list of facts

In [53]:
response = await Runner.run(agent, "Hi there. My name is Bilal.")
print(response.final_output)

Hello Bilal! Nice to meet you. How can I assist you today? Is there anything specific you'd like to discuss or any questions you have?


In [54]:
response.to_input_list()

[{'content': 'Hi there. My name is Bilal.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Bilal! Nice to meet you. How can I assist you today? Is there anything specific you'd like to discuss or any questions you have?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-569'}}]

In [55]:
next_input =  response.to_input_list() + [{"role": "user", "content": "What's my name?"}]

next_input

[{'content': 'Hi there. My name is Bilal.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Bilal! Nice to meet you. How can I assist you today? Is there anything specific you'd like to discuss or any questions you have?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-569'}},
 {'role': 'user', 'content': "What's my name?"}]

In [56]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Bilal.


### Another approach - use OpenAI Agents SDK built in SQLLite session

In [57]:
session = SQLiteSession("123456")

In [58]:
response = await Runner.run(agent, "Hi there. My name is Bilal.", session=session)
print(response.final_output)

Hello Bilal! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?


In [59]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Bilal.
